In [0]:
CREATE TABLE cdfcatalog.schema1.sales_transactions (
    id INT PRIMARY KEY,
    emp_id INT,
    emp_name VARCHAR(50),
    department VARCHAR(50),
    region VARCHAR(50),
    sale_date DATE,
    amount INT
);

In [0]:
INSERT INTO cdfcatalog.schema1.sales_transactions VALUES
(1, 101, 'Ayesha', 'Sales', 'North', '2026-01-01', 500),
(2, 102, 'Ali',    'Sales', 'North', '2026-01-02', 700),
(3, 103, 'Sara',   'Sales', 'South', '2026-01-02', 700),
(4, 104, 'John',   'IT',    'North', '2026-01-03', 300),
(5, 105, 'Emma',   'IT',    'South', '2026-01-03', 900),

(6, 101, 'Ayesha', 'Sales', 'North', '2026-01-04', 400),
(7, 102, 'Ali',    'Sales', 'North', '2026-01-05', 800),
(8, 103, 'Sara',   'Sales', 'South', '2026-01-05', 200),
(9, 104, 'John',   'IT',    'North', '2026-01-06', 600),
(10,105, 'Emma',   'IT',    'South', '2026-01-06', 300),

(11,101, 'Ayesha', 'Sales', 'North', '2026-01-07', 700),
(12,102, 'Ali',    'Sales', 'North', '2026-01-07', 500),
(13,103, 'Sara',   'Sales', 'South', '2026-01-08', 900),
(14,104, 'John',   'IT',    'North', '2026-01-08', 1000),
(15,105, 'Emma',   'IT',    'South', '2026-01-09', 1000),

(16,101, 'Ayesha', 'Sales', 'North', '2026-01-10', 300),
(17,102, 'Ali',    'Sales', 'North', '2026-01-10', 200),
(18,103, 'Sara',   'Sales', 'South', '2026-01-11', 400),
(19,104, 'John',   'IT',    'North', '2026-01-11', 500),
(20,105, 'Emma',   'IT',    'South', '2026-01-12', 600);

In [0]:
-- when over() with no partition clause it consider whole table as a single window and perform aggregation on whole data
SELECT *,AVG(amount) OVER() FROM cdfcatalog.schema1.sales_transactions

id,emp_id,emp_name,department,region,sale_date,amount,AVG(amount)OVER()
1,101,Ayesha,Sales,North,2026-01-01,500,575.0
2,102,Ali,Sales,North,2026-01-02,700,575.0
3,103,Sara,Sales,South,2026-01-02,700,575.0
4,104,John,IT,North,2026-01-03,300,575.0
5,105,Emma,IT,South,2026-01-03,900,575.0
6,101,Ayesha,Sales,North,2026-01-04,400,575.0
7,102,Ali,Sales,North,2026-01-05,800,575.0
8,103,Sara,Sales,South,2026-01-05,200,575.0
9,104,John,IT,North,2026-01-06,600,575.0
10,105,Emma,IT,South,2026-01-06,300,575.0


In [0]:
SELECT *,AVG(amount) OVER(partition by department) as avg_amount_window_on_dept FROM cdfcatalog.schema1.sales_transactions

id,emp_id,emp_name,department,region,sale_date,amount,avg_amount_window_on_dept
1,101,Ayesha,Sales,North,2026-01-01,500,525.0
2,102,Ali,Sales,North,2026-01-02,700,525.0
3,103,Sara,Sales,South,2026-01-02,700,525.0
4,104,John,IT,North,2026-01-03,300,650.0
5,105,Emma,IT,South,2026-01-03,900,650.0
6,101,Ayesha,Sales,North,2026-01-04,400,525.0
7,102,Ali,Sales,North,2026-01-05,800,525.0
8,103,Sara,Sales,South,2026-01-05,200,525.0
9,104,John,IT,North,2026-01-06,600,650.0
10,105,Emma,IT,South,2026-01-06,300,650.0


In [0]:
SELECT *,MIN(amount) OVER() ,MAX(amount) OVER() FROM cdfcatalog.schema1.sales_transactions

id,emp_id,emp_name,department,region,sale_date,amount,MIN(amount)OVER(),MAX(amount)OVER()
1,101,Ayesha,Sales,North,2026-01-01,500,200,1000
2,102,Ali,Sales,North,2026-01-02,700,200,1000
3,103,Sara,Sales,South,2026-01-02,700,200,1000
4,104,John,IT,North,2026-01-03,300,200,1000
5,105,Emma,IT,South,2026-01-03,900,200,1000
6,101,Ayesha,Sales,North,2026-01-04,400,200,1000
7,102,Ali,Sales,North,2026-01-05,800,200,1000
8,103,Sara,Sales,South,2026-01-05,200,200,1000
9,104,John,IT,North,2026-01-06,600,200,1000
10,105,Emma,IT,South,2026-01-06,300,200,1000


In [0]:
SELECT *,
ROW_NUMBER() OVER (PARTITION BY emp_id ORDER BY sale_date) AS rn
FROM cdfcatalog.schema1.sales_transactions;

id,emp_id,emp_name,department,region,sale_date,amount,rn
1,101,Ayesha,Sales,North,2026-01-01,500,1
6,101,Ayesha,Sales,North,2026-01-04,400,2
11,101,Ayesha,Sales,North,2026-01-07,700,3
16,101,Ayesha,Sales,North,2026-01-10,300,4
2,102,Ali,Sales,North,2026-01-02,700,1
7,102,Ali,Sales,North,2026-01-05,800,2
12,102,Ali,Sales,North,2026-01-07,500,3
17,102,Ali,Sales,North,2026-01-10,200,4
3,103,Sara,Sales,South,2026-01-02,700,1
8,103,Sara,Sales,South,2026-01-05,200,2


In [0]:
-- to see only those sales where amount is greater than average amount of that department
SELECT * FROM (SELECT *,AVG(amount) OVER(PARTITION BY department) as avg_amount_window_on_dept 
FROM cdfcatalog.schema1.sales_transactions) as t
WHERE t.amount > t.avg_amount_window_on_dept;

id,emp_id,emp_name,department,region,sale_date,amount,avg_amount_window_on_dept
2,102,Ali,Sales,North,2026-01-02,700,525.0
3,103,Sara,Sales,South,2026-01-02,700,525.0
5,105,Emma,IT,South,2026-01-03,900,650.0
7,102,Ali,Sales,North,2026-01-05,800,525.0
11,101,Ayesha,Sales,North,2026-01-07,700,525.0
13,103,Sara,Sales,South,2026-01-08,900,525.0
14,104,John,IT,North,2026-01-08,1000,650.0
15,105,Emma,IT,South,2026-01-09,1000,650.0


In [0]:
--  RANK() window function , rank to sale who has larger amount
SELECT *,RANK() OVER(order by amount desc) FROM cdfcatalog.schema1.sales_transactions

id,emp_id,emp_name,department,region,sale_date,amount,RANK()OVER(ORDERBYamountDESC)
14,104,John,IT,North,2026-01-08,1000,1
15,105,Emma,IT,South,2026-01-09,1000,1
13,103,Sara,Sales,South,2026-01-08,900,3
5,105,Emma,IT,South,2026-01-03,900,3
7,102,Ali,Sales,North,2026-01-05,800,5
3,103,Sara,Sales,South,2026-01-02,700,6
11,101,Ayesha,Sales,North,2026-01-07,700,6
2,102,Ali,Sales,North,2026-01-02,700,6
20,105,Emma,IT,South,2026-01-12,600,9
9,104,John,IT,North,2026-01-06,600,9


In [0]:
--  RANK() window function , rank to sale who has larger amount in each department
SELECT *,RANK() OVER(PARTITION BY department order by amount desc) as rank FROM cdfcatalog.schema1.sales_transactions

id,emp_id,emp_name,department,region,sale_date,amount,rank
14,104,John,IT,North,2026-01-08,1000,1
15,105,Emma,IT,South,2026-01-09,1000,1
5,105,Emma,IT,South,2026-01-03,900,3
9,104,John,IT,North,2026-01-06,600,4
20,105,Emma,IT,South,2026-01-12,600,4
19,104,John,IT,North,2026-01-11,500,6
4,104,John,IT,North,2026-01-03,300,7
10,105,Emma,IT,South,2026-01-06,300,7
13,103,Sara,Sales,South,2026-01-08,900,1
7,102,Ali,Sales,North,2026-01-05,800,2


In [0]:
--  DENSE_RANK() window function , rank to sale who has larger amount in each department
SELECT *,DENSE_RANK() OVER(PARTITION BY department order by amount desc) as rank FROM cdfcatalog.schema1.sales_transactions

id,emp_id,emp_name,department,region,sale_date,amount,rank
14,104,John,IT,North,2026-01-08,1000,1
15,105,Emma,IT,South,2026-01-09,1000,1
5,105,Emma,IT,South,2026-01-03,900,2
9,104,John,IT,North,2026-01-06,600,3
20,105,Emma,IT,South,2026-01-12,600,3
19,104,John,IT,North,2026-01-11,500,4
4,104,John,IT,North,2026-01-03,300,5
10,105,Emma,IT,South,2026-01-06,300,5
13,103,Sara,Sales,South,2026-01-08,900,1
7,102,Ali,Sales,North,2026-01-05,800,2


In [0]:
--  ROW_Number() window function , assighn rownumber to  sale who has larger amount in each department
SELECT *,ROW_NUMBER() OVER(ORDER by amount desc) as rownummber FROM cdfcatalog.schema1.sales_transactions

id,emp_id,emp_name,department,region,sale_date,amount,rownummber
14,104,John,IT,North,2026-01-08,1000,1
15,105,Emma,IT,South,2026-01-09,1000,2
13,103,Sara,Sales,South,2026-01-08,900,3
5,105,Emma,IT,South,2026-01-03,900,4
7,102,Ali,Sales,North,2026-01-05,800,5
3,103,Sara,Sales,South,2026-01-02,700,6
11,101,Ayesha,Sales,North,2026-01-07,700,7
2,102,Ali,Sales,North,2026-01-02,700,8
20,105,Emma,IT,South,2026-01-12,600,9
9,104,John,IT,North,2026-01-06,600,10


In [0]:
--to see which top 2 employees perofrmr more sales

SElECT * FROM (SElECT month(sale_date) as monthh ,emp_id, RANK() OVER(PArtition by monthh order by sum(amount) desc)
FROM cdfcatalog.schema1.sales_transactions
GROUP BY monthh ,emp_id) as t 
WHERE rank <=2

In [0]:
-- SELECT *,first_value(emp_name) OVER (order by amount desc) FROM cdfcatalog.schema1.sales_transactions;

-- last_value concept related to frrames
SELECT *,last_value(emp_name) OVER (order by amount desc) FROM cdfcatalog.schema1.sales_transactions;


SELECT *,NTH_VALUE(emp_name) OVER (order by amount desc) FROM cdfcatalog.schema1.sales_transactions

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7633134112304278>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', '-- SELECT *,first_value(emp_name) OVER (order by amount desc) FROM cdfcatalog.schema1.sales_transactions;\n\n-- last_value concept related to frrames\nSELECT *,last_value(emp_name) OVER (order by amount desc) FROM cdfcatalog.schema1.sales_transactions;\n\n\nSELECT *,NTH_VALUE(emp_name) OVER (order by amount desc) FROM cdfcatalog.schema1.sales_transactions\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_sil

In [0]:
SELECT * ,lag (amount) OVER (order by emP_id desc) FROM cdfcatalog.schema1.sales_transactions;

id,emp_id,emp_name,department,region,sale_date,amount,lag(amount)OVER(ORDERBYemP_idDESC)
5,105,Emma,IT,South,2026-01-03,900,null
10,105,Emma,IT,South,2026-01-06,300,900
15,105,Emma,IT,South,2026-01-09,1000,300
20,105,Emma,IT,South,2026-01-12,600,1000
4,104,John,IT,North,2026-01-03,300,600
9,104,John,IT,North,2026-01-06,600,300
14,104,John,IT,North,2026-01-08,1000,600
19,104,John,IT,North,2026-01-11,500,1000
3,103,Sara,Sales,South,2026-01-02,700,500
8,103,Sara,Sales,South,2026-01-05,200,700


In [0]:
SELECT * ,lead (amount) OVER (order by emP_id desc) FROM cdfcatalog.schema1.sales_transactions;

id,emp_id,emp_name,department,region,sale_date,amount,lead(amount)OVER(ORDERBYemP_idDESC)
5,105,Emma,IT,South,2026-01-03,900,300
10,105,Emma,IT,South,2026-01-06,300,1000
15,105,Emma,IT,South,2026-01-09,1000,600
20,105,Emma,IT,South,2026-01-12,600,300
4,104,John,IT,North,2026-01-03,300,600
9,104,John,IT,North,2026-01-06,600,1000
14,104,John,IT,North,2026-01-08,1000,500
19,104,John,IT,North,2026-01-11,500,700
3,103,Sara,Sales,South,2026-01-02,700,200
8,103,Sara,Sales,South,2026-01-05,200,900
